# Mamba SCA — Training & Evaluation Notebook

**Paper**: "Deep Learning-based Side-Channel Attack: Mamba Approach"  
**Author**: Beatrise Bertule, Radboud University Nijmegen, January 2026

This notebook trains the paper-faithful Mamba model on **ASCADv1** and evaluates it with a full key-recovery attack.

---

### Hyperparameters (Tables 4.3, 4.4)

| Parameter | Value | Source |
|-----------|-------|--------|
| Batch size | 768 | Table 4.4 |
| Learning rate | 5×10⁻³ | Table 4.4 |
| Optimizer | Adam + weight decay | Section 4.4.2 |
| Max epochs | 100 | Section 4.4.2 |
| Loss | CrossEntropyLoss | Section 4.3.1 |
| Validation metric | Mean GE (100 runs × 1000 traces) | Section 4.4.2 |
| Target byte | 2 (ASCADv1) | Section 4.4.1 |

## 1 · Imports & Configuration

In [ ]:
import os
import sys
import time
import math
import h5py
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# -- Make the package importable from the notebook --
PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from mamba_sca_model import MambaSCAModel

# ── Configuration ──────────────────────────────────────────────────────
DATA_PATH    = os.path.join(PROJECT_DIR, 'data', 'ASCAD.h5')
SAVE_PATH    = os.path.join(PROJECT_DIR, 'best_model.pt')
TARGET_BYTE  = 2          # ASCADv1 (Section 4.4.1)
BATCH_SIZE   = 768        # Table 4.4
LR           = 5e-3       # Table 4.4
WEIGHT_DECAY = 1e-4       # Paper: "weight decay enabled"
MAX_EPOCHS   = 100        # Section 4.4.2
N_GE_RUNS    = 100        # Section 4.4.2: 100 attack simulations
N_GE_TRACES  = 1000       # Section 4.4.3: 1,000 traces per simulation
LOG_EVERY_N_STEPS = 10    # Print train loss every N mini-batch steps

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2 · AES S-box & Label Helpers

In [ ]:
# AES S-box lookup table (Section 2.2.2, Table 2.1)
AES_SBOX = np.array([
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16,
], dtype=np.uint8)


def make_labels(plaintexts, key, target_byte):
    """Identity leakage model: y = Sbox[pt[i] XOR k[i]] (Eq. 4.1/4.2)"""
    intermediate = plaintexts[:, target_byte] ^ key[target_byte]
    return AES_SBOX[intermediate].astype(np.int64)


print("S-box and label helpers ready.")

## 3 · Load ASCADv1 Dataset

In [ ]:
print(f"Loading dataset from: {DATA_PATH}")
print("This may take a moment...\n")

with h5py.File(DATA_PATH, 'r') as f:
    # ── Print dataset structure ────────────────────────────────────
    print("HDF5 structure:")
    def print_structure(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: shape={obj.shape}, dtype={obj.dtype}")
    f.visititems(print_structure)
    print()

    # ── Profiling (train) traces  ──────────────────────────────────
    X_profiling = np.array(f['Profiling_traces/traces'], dtype=np.float32)
    pt_profiling = np.array(f['Profiling_traces/metadata']['plaintext'])
    key_profiling = np.array(f['Profiling_traces/metadata']['key'])

    # ── Attack (test) traces ──────────────────────────────────────
    X_attack = np.array(f['Attack_traces/traces'], dtype=np.float32)
    pt_attack = np.array(f['Attack_traces/metadata']['plaintext'])
    key_attack = np.array(f['Attack_traces/metadata']['key'])

print(f"Profiling traces : {X_profiling.shape}")
print(f"Attack traces    : {X_attack.shape}")
print(f"Trace length     : {X_profiling.shape[1]}")

# ── Extract the correct key byte (for GE computation) ─────────────
correct_key_byte = int(key_profiling[0][TARGET_BYTE])
print(f"\nCorrect key byte (byte {TARGET_BYTE}): 0x{correct_key_byte:02X} ({correct_key_byte})")

## 4 · Prepare Train / Validation / Test Splits

In [ ]:
# ── Data splits (Table 4.3) ───────────────────────────────────────────
#   Train : first 40,000 profiling traces
#   Val   : next  10,000 profiling traces
#   Test  : all   10,000 attack traces

N_TRAIN = 40000
N_VAL   = 10000

X_train = X_profiling[:N_TRAIN]
X_val   = X_profiling[N_TRAIN:N_TRAIN + N_VAL]
X_test  = X_attack

pt_train = pt_profiling[:N_TRAIN]
pt_val   = pt_profiling[N_TRAIN:N_TRAIN + N_VAL]
pt_test  = pt_attack

key_train = key_profiling[:N_TRAIN]
key_val   = key_profiling[N_TRAIN:N_TRAIN + N_VAL]

# ── Compute labels (Identity leakage model, Eq. 4.1/4.2) ─────────────
y_train = make_labels(pt_train, key_train[0], TARGET_BYTE)
y_val   = make_labels(pt_val,   key_val[0],   TARGET_BYTE)

print(f"Train : {X_train.shape[0]:>6,} traces  |  labels range: [{y_train.min()}, {y_train.max()}]")
print(f"Val   : {X_val.shape[0]:>6,} traces  |  labels range: [{y_val.min()}, {y_val.max()}]")
print(f"Test  : {X_test.shape[0]:>6,} traces  (used only for attack evaluation)")

# Quick sanity check: 256-class distribution should be roughly uniform
unique, counts = np.unique(y_train, return_counts=True)
print(f"\nLabel distribution (train): {len(unique)} classes, "
      f"min={counts.min()}, max={counts.max()}, mean={counts.mean():.1f}")

## 5 · Normalise & Build DataLoaders

In [ ]:
# ── StandardScaler fitted on TRAIN set only (Section 4.4.1) ───────────
train_mean = X_train.mean(axis=0, keepdims=True)          # [1, T]
train_std  = X_train.std(axis=0, keepdims=True) + 1e-8    # [1, T]

X_train_n = ((X_train - train_mean) / train_std).astype(np.float32)
X_val_n   = ((X_val   - train_mean) / train_std).astype(np.float32)

print(f"Normalisation stats  — mean range: [{train_mean.min():.4f}, {train_mean.max():.4f}]")
print(f"                     — std  range: [{train_std.min():.4f},  {train_std.max():.4f}]")

# ── Convert to tensors ─────────────────────────────────────────────────
X_train_t = torch.from_numpy(X_train_n)
y_train_t = torch.from_numpy(y_train)
X_val_t   = torch.from_numpy(X_val_n)
y_val_t   = torch.from_numpy(y_val)

# ── DataLoaders ────────────────────────────────────────────────────────
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(X_val_t, y_val_t),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

total_train_steps = math.ceil(len(X_train_t) / BATCH_SIZE)
total_val_steps   = math.ceil(len(X_val_t)   / BATCH_SIZE)
print(f"\nTrain batches per epoch: {total_train_steps}")
print(f"Val   batches per epoch: {total_val_steps}")

## 6 · Build Model

In [ ]:
trace_length = X_train.shape[1]

model = MambaSCAModel(
    trace_length = trace_length,
    d_model      = 64,      # Table 4.2
    n_blocks     = 2,       # Table 4.2
    num_classes  = 256,
    d_state      = 8,       # ~148k params to match paper
    d_conv       = 4,       # standard Mamba default
    expand       = 2,       # standard Mamba default
)

model.to(DEVICE)
total_params = model.count_parameters()

print(f"Model on: {DEVICE}")
print(f"Trace length: {trace_length}")
print(f"Total trainable parameters: {total_params:,}")
print(f"Paper reports: ~148,000")
print(f"Match: {'✅ YES' if 140_000 <= total_params <= 160_000 else '⚠️ Close' if 100_000 <= total_params <= 200_000 else '❌ NO'}")
print(f"\nArchitecture:\n{model}")

## 7 · Guessing Entropy Helper

In [ ]:
def compute_guessing_entropy(
    model, traces, plaintexts, correct_key, target_byte,
    n_runs=100, n_traces=1000, device='cpu', eps=1e-10,
):
    """
    Mean GE across n_runs attack simulations (Eq. 2.25).
    Returns the mean rank of the correct key.
    """
    model.eval()
    N = traces.shape[0]

    # Get all model predictions
    all_probs = []
    with torch.no_grad():
        loader = DataLoader(TensorDataset(traces), batch_size=512, shuffle=False)
        for (batch,) in loader:
            logits = model(batch.to(device))
            probs  = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    all_probs = np.concatenate(all_probs, axis=0)  # [N, 256]

    # Hypothetical labels for all key candidates
    pt_byte = plaintexts[:, target_byte].astype(np.uint8)
    hyp_labels = AES_SBOX[
        pt_byte[:, None] ^ np.arange(256, dtype=np.uint8)[None, :]
    ]  # [N, 256]

    ranks = []
    for _ in range(n_runs):
        idx = np.random.choice(N, size=n_traces, replace=False)
        probs_sub = all_probs[idx]
        hyp_sub   = hyp_labels[idx]

        scores = np.zeros(256, dtype=np.float64)
        for k in range(256):
            log_probs = np.log(probs_sub[np.arange(n_traces), hyp_sub[:, k]] + eps)
            scores[k] = log_probs.sum()

        rank = np.where(np.argsort(scores)[::-1] == correct_key)[0][0]
        ranks.append(rank)

    return float(np.mean(ranks))


print("Guessing Entropy function ready.")

## 8 · Training Loop

The training loop logs:
- **Per-step**: training loss every `LOG_EVERY_N_STEPS` mini-batches
- **Per-epoch**: average training loss, average validation loss, validation GE
- **Best checkpoint**: saved when validation GE improves
- **Live plot**: training loss & validation loss curves updated each epoch

In [ ]:
# ── Setup optimizer & loss ─────────────────────────────────────────────
optimizer = torch.optim.Adam(
    model.parameters(),
    lr           = LR,
    weight_decay = WEIGHT_DECAY,
)
criterion = nn.CrossEntropyLoss()

# ── History tracking ───────────────────────────────────────────────────
history = {
    'train_loss': [],
    'val_loss':   [],
    'val_ge':     [],
}
step_losses = []  # (global_step, loss) for per-step plot

best_ge    = float('inf')
best_epoch = -1
global_step = 0

print("=" * 75)
print(f"  TRAINING START  |  {MAX_EPOCHS} epochs  |  {total_train_steps} steps/epoch")
print(f"  Batch size: {BATCH_SIZE}  |  LR: {LR}  |  Device: {DEVICE}")
print("=" * 75)
print()

training_start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()

    # ─── TRAINING PHASE ────────────────────────────────────────────
    model.train()
    epoch_train_loss = 0.0
    epoch_correct    = 0
    epoch_total      = 0

    for step, (X_batch, y_batch) in enumerate(train_loader, 1):
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        batch_size = X_batch.size(0)
        epoch_train_loss += batch_loss * batch_size
        epoch_total      += batch_size

        # Track predictions for accuracy
        preds = logits.argmax(dim=-1)
        epoch_correct += (preds == y_batch).sum().item()

        # Track step-level loss
        global_step += 1
        step_losses.append((global_step, batch_loss))

        # ── Per-step logging ───────────────────────────────────────
        if step % LOG_EVERY_N_STEPS == 0 or step == total_train_steps:
            running_avg = epoch_train_loss / epoch_total
            print(
                f"  [Epoch {epoch:3d}/{MAX_EPOCHS}] "
                f"Step {step:3d}/{total_train_steps} | "
                f"batch_loss: {batch_loss:.4f} | "
                f"running_avg: {running_avg:.4f}"
            )

    avg_train_loss = epoch_train_loss / epoch_total
    train_acc      = epoch_correct / epoch_total

    # ─── VALIDATION PHASE (loss) ───────────────────────────────────
    model.eval()
    val_loss_total = 0.0
    val_correct    = 0
    val_total      = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            logits  = model(X_batch)
            loss    = criterion(logits, y_batch)

            val_loss_total += loss.item() * X_batch.size(0)
            val_total      += X_batch.size(0)

            preds = logits.argmax(dim=-1)
            val_correct += (preds == y_batch).sum().item()

    avg_val_loss = val_loss_total / val_total
    val_acc      = val_correct / val_total

    # ─── VALIDATION GE (Section 4.4.2: the ONLY metric for model selection) ──
    val_ge = compute_guessing_entropy(
        model       = model,
        traces      = X_val_t,
        plaintexts  = pt_val,
        correct_key = correct_key_byte,
        target_byte = TARGET_BYTE,
        n_runs      = N_GE_RUNS,
        n_traces    = N_GE_TRACES,
        device      = DEVICE,
    )

    # ── Record history ──────────────────────────────────────────────
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_ge'].append(val_ge)

    epoch_time = time.time() - epoch_start
    elapsed    = time.time() - training_start_time

    # ── Epoch summary ──────────────────────────────────────────────
    print()
    print(f"{'─' * 75}")
    print(f"  Epoch {epoch:3d}/{MAX_EPOCHS} complete in {epoch_time:.1f}s  "
          f"(total elapsed: {elapsed/60:.1f} min)")
    print(f"  Train Loss : {avg_train_loss:.4f}   |   Train Acc : {train_acc:.4f}")
    print(f"  Val   Loss : {avg_val_loss:.4f}   |   Val   Acc : {val_acc:.4f}")
    print(f"  Val   GE   : {val_ge:.4f}  ", end='')

    # ── Trend indicators ───────────────────────────────────────────
    if len(history['train_loss']) >= 2:
        tl_delta = history['train_loss'][-1] - history['train_loss'][-2]
        vl_delta = history['val_loss'][-1]   - history['val_loss'][-2]
        ge_delta = history['val_ge'][-1]     - history['val_ge'][-2]
        print(f"  (ΔGE: {ge_delta:+.2f})")
        print(f"  Trends → train_loss: {tl_delta:+.4f}  |  "
              f"val_loss: {vl_delta:+.4f}  |  val_GE: {ge_delta:+.2f}")
    else:
        print()

    # ── Checkpoint: save best GE model ─────────────────────────────
    if val_ge < best_ge:
        best_ge    = val_ge
        best_epoch = epoch
        torch.save({
            'epoch'       : best_epoch,
            'model_state' : model.state_dict(),
            'val_ge'      : best_ge,
            'mean_std'    : (train_mean, train_std),
            'history'     : history,
        }, SAVE_PATH)
        print(f"  ★ NEW BEST — saved checkpoint (val_GE={best_ge:.4f} at epoch {best_epoch})")
    else:
        gap = epoch - best_epoch
        print(f"  No improvement for {gap} epoch(s) (best: {best_ge:.4f} @ epoch {best_epoch})")

    print(f"{'─' * 75}")
    print()

# ── Final summary ──────────────────────────────────────────────────────
total_time = time.time() - training_start_time
print("\n" + "=" * 75)
print(f"  TRAINING COMPLETE")
print(f"  Total time  : {total_time/60:.1f} minutes")
print(f"  Best val GE : {best_ge:.4f} at epoch {best_epoch}")
print(f"  Final train loss : {history['train_loss'][-1]:.4f}")
print(f"  Final val   loss : {history['val_loss'][-1]:.4f}")
print(f"  Checkpoint saved : {SAVE_PATH}")
print("=" * 75)

## 9 · Training Curves

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Train vs Val Loss ──────────────────────────────────────────────────
axes[0].plot(epochs_range, history['train_loss'], 'b-o', markersize=3, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', markersize=3, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CrossEntropy Loss')
axes[0].set_title('Training vs Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Validation GE ──────────────────────────────────────────────────────
axes[1].plot(epochs_range, history['val_ge'], 'g-o', markersize=3, label='Val GE')
axes[1].axhline(y=best_ge, color='green', linestyle='--', alpha=0.5, label=f'Best: {best_ge:.2f}')
axes[1].axvline(x=best_epoch, color='orange', linestyle='--', alpha=0.5, label=f'Best epoch: {best_epoch}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Guessing Entropy')
axes[1].set_title('Validation GE (Model Selection Metric)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ── Step-level training loss ───────────────────────────────────────────
steps, losses = zip(*step_losses)
axes[2].plot(steps, losses, 'b-', alpha=0.3, linewidth=0.5)
# Moving average for clarity
window = min(50, len(losses))
if window > 1:
    moving_avg = np.convolve(losses, np.ones(window)/window, mode='valid')
    axes[2].plot(steps[window-1:], moving_avg, 'b-', linewidth=2, label=f'MA-{window}')
axes[2].set_xlabel('Global Step')
axes[2].set_ylabel('Batch Loss')
axes[2].set_title('Per-Step Training Loss')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_curves.png")

## 10 · Load Best Checkpoint for Evaluation

In [ ]:
print(f"Loading best checkpoint from: {SAVE_PATH}")

checkpoint = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state'])
model.to(DEVICE)
model.eval()

ckpt_mean, ckpt_std = checkpoint['mean_std']

print(f"Loaded epoch {checkpoint['epoch']} with val_GE = {checkpoint['val_ge']:.4f}")

## 11 · Attack Phase Evaluation (Section 4.4.3)

In [ ]:
print("Running full attack evaluation on 10,000 test traces...")
print("(100 simulations × 1,000 traces each + convergence curve)\n")

eval_start = time.time()

# ── Normalise test set with TRAIN statistics ──────────────────────────
X_test_n = ((X_test - ckpt_mean) / ckpt_std).astype(np.float32)
X_test_t = torch.from_numpy(X_test_n)

N_test = X_test_t.shape[0]
eps    = 1e-10

# ── Get all predictions ───────────────────────────────────────────────
print("Computing model predictions...")
all_probs = []
with torch.no_grad():
    loader = DataLoader(TensorDataset(X_test_t), batch_size=512, shuffle=False)
    for (batch,) in loader:
        logits = model(batch.to(DEVICE))
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
all_probs = np.concatenate(all_probs, axis=0)  # [N, 256]

# ── Hypothetical labels ───────────────────────────────────────────────
pt_byte_test = pt_test[:, TARGET_BYTE].astype(np.uint8)
hyp_labels   = AES_SBOX[
    pt_byte_test[:, None] ^ np.arange(256, dtype=np.uint8)[None, :]
]  # [N, 256]

# ── 1. Key rank on full 10,000 traces ─────────────────────────────────
print("Computing key rank on all 10,000 traces...")
scores_full = np.zeros(256, dtype=np.float64)
for k in range(256):
    scores_full[k] = np.log(all_probs[np.arange(N_test), hyp_labels[:, k]] + eps).sum()
key_rank = int(np.where(np.argsort(scores_full)[::-1] == correct_key_byte)[0][0])
print(f"  Key rank (10k traces): {key_rank}")

# ── 2. Mean GE + Median GE over 100 runs ─────────────────────────────
print("Running 100 attack simulations...")
attack_ranks = []
for run in range(100):
    idx = np.random.choice(N_test, size=1000, replace=False)
    scores = np.zeros(256, dtype=np.float64)
    for k in range(256):
        scores[k] = np.log(all_probs[idx][np.arange(1000), hyp_labels[idx, k]] + eps).sum()
    rank = int(np.where(np.argsort(scores)[::-1] == correct_key_byte)[0][0])
    attack_ranks.append(rank)
    if (run + 1) % 20 == 0:
        print(f"  Completed {run+1}/100 runs (current mean GE: {np.mean(attack_ranks):.2f})")

mean_ge   = float(np.mean(attack_ranks))
median_ge = float(np.median(attack_ranks))

# ── 3. GE convergence curve ───────────────────────────────────────────
print("Computing GE convergence curve...")
conv_ranks = []
for run in range(100):
    idx = np.random.choice(N_test, size=1000, replace=False)
    trace_ranks = []
    scores = np.zeros(256, dtype=np.float64)
    for t in range(1000):
        k_idx = hyp_labels[idx[t], :]
        log_p = np.log(all_probs[idx[t], k_idx] + eps)
        scores += log_p
        rank = int(np.where(np.argsort(scores)[::-1] == correct_key_byte)[0][0])
        trace_ranks.append(rank)
    conv_ranks.append(trace_ranks)
    if (run + 1) % 20 == 0:
        print(f"  Convergence: {run+1}/100 runs done")

ge_convergence = np.mean(conv_ranks, axis=0)  # [1000]
ge_conv_value  = float(ge_convergence[-1])
ge_conv_trace  = int(np.argmax(ge_convergence <= 1.0)) if (ge_convergence <= 1.0).any() else 1000

eval_time = time.time() - eval_start

# ── Results ────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("  ATTACK EVALUATION RESULTS")
print("=" * 60)
print(f"  Key Rank (full 10k traces) : {key_rank}")
print(f"  Mean GE   (100 runs)       : {mean_ge:.4f}")
print(f"  Median GE (100 runs)       : {median_ge:.4f}")
print(f"  GE Convergence value       : {ge_conv_value:.4f}")
print(f"  Traces to GE ≤ 1           : {ge_conv_trace}")
print(f"  Evaluation time            : {eval_time:.1f}s")
print("=" * 60)

if key_rank == 0:
    print("\n🎉 Key rank = 0 — the correct key is the TOP candidate!")
elif key_rank <= 5:
    print(f"\n✅ Key rank = {key_rank} — very close, practically broken.")
else:
    print(f"\n⚠️ Key rank = {key_rank} — model may need more training.")

## 12 · GE Convergence Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── GE Convergence over traces ─────────────────────────────────────────
axes[0].plot(range(1, 1001), ge_convergence, 'b-', linewidth=1.5)
axes[0].axhline(y=1.0, color='red', linestyle='--', alpha=0.6, label='GE = 1 threshold')
axes[0].axhline(y=0.0, color='green', linestyle='--', alpha=0.6, label='GE = 0 (perfect)')
if ge_conv_trace < 1000:
    axes[0].axvline(x=ge_conv_trace, color='orange', linestyle='--', alpha=0.6,
                    label=f'GE ≤ 1 at trace {ge_conv_trace}')
axes[0].set_xlabel('Number of Attack Traces')
axes[0].set_ylabel('Mean Guessing Entropy')
axes[0].set_title('GE Convergence Curve (100 simulations avg)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 1000)

# ── Attack rank distribution ───────────────────────────────────────────
axes[1].hist(attack_ranks, bins=range(0, max(attack_ranks) + 2), 
             color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(x=mean_ge, color='red', linestyle='--', linewidth=2,
                label=f'Mean GE = {mean_ge:.2f}')
axes[1].axvline(x=median_ge, color='orange', linestyle='--', linewidth=2,
                label=f'Median GE = {median_ge:.2f}')
axes[1].set_xlabel('Key Rank')
axes[1].set_ylabel('Count')
axes[1].set_title('Attack Rank Distribution (100 runs)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: evaluation_results.png")

## 13 · Summary Table

In [ ]:
summary = f"""
╔══════════════════════════════════════════════════════════════╗
║                    EXPERIMENT SUMMARY                       ║
╠══════════════════════════════════════════════════════════════╣
║  Model             : MambaSCAModel                          ║
║  Parameters         : {total_params:>10,}                          ║
║  Dataset            : ASCADv1 (byte {TARGET_BYTE})                    ║
║  Trace length       : {trace_length:>10}                          ║
║──────────────────────────────────────────────────────────────║
║  Training                                                    ║
║    Epochs trained    : {MAX_EPOCHS:>10}                          ║
║    Best epoch        : {best_epoch:>10}                          ║
║    Best val GE       : {best_ge:>10.4f}                          ║
║    Final train loss  : {history['train_loss'][-1]:>10.4f}                          ║
║    Final val   loss  : {history['val_loss'][-1]:>10.4f}                          ║
║──────────────────────────────────────────────────────────────║
║  Attack Evaluation                                           ║
║    Key rank (10k)    : {key_rank:>10}                          ║
║    Mean GE           : {mean_ge:>10.4f}                          ║
║    Median GE         : {median_ge:>10.4f}                          ║
║    Traces to GE ≤ 1  : {ge_conv_trace:>10}                          ║
╚══════════════════════════════════════════════════════════════╝
"""
print(summary)